In [10]:
import sys
import os

sys.path.append(os.path.dirname(os.path.abspath(os.getcwd())))

In [11]:
from src.mslm.models import Imitator, PositionalEncoding

In [12]:
import torch
from torch.export import export, Dim
import coremltools as ct

In [13]:
torch.serialization.add_safe_globals([Imitator, PositionalEncoding])

In [14]:
class ExportWrapper(torch.nn.Module):
    def __init__(self, imitator: Imitator):
        super().__init__()
        self.imitator = imitator

    def forward(self, x: torch.Tensor, frames_padding_mask: torch.Tensor):
        return self.imitator(x, frames_padding_mask)

In [15]:
model_path = "/home/giorgio6846/Code/Sign-AI/outputs/checkpoints/85/7/9/"
mobile_path = "/home/giorgio6846/Code/Sign-AI/outputs/mobile/"


dynamic_shapes = {
    "x": {1: Dim("seq_len", min=1, max=4000)},
    "frames_padding_mask": {1: Dim("seq_len", min=1, max=4000)},
}

model = torch.load(os.path.join(model_path, "model.pt"), map_location="cpu", weights_only=False)
model.eval()

wrapper = ExportWrapper(model)
wrapper.eval()

#exported_model = export(wrapper, 
#                (torch.randn(1, 60, 133, 2), torch.zeros(1, 60)), 
#                dynamic_shapes=dynamic_shapes
#)

scripted_wrapper = torch.jit.script(wrapper)

mlmodel = ct.convert(
    scripted_wrapper, 
    minimum_deployment_target=ct.target.iOS16,
    inputs=[
        ct.TensorType(name="x", shape=(1, ct.RangeDim(lower_bound=1, upper_bound=4000), 133, 2)),
        ct.TensorType(name="frames_padding_mask", shape=(1, ct.RangeDim(lower_bound=1, upper_bound=4000)))
    ],
)
mlmodel.save(os.path.join(mobile_path, "model.mlmodel"))

UnsupportedNodeError: function definitions aren't supported:
  File "/home/giorgio6846/Code/Sign-AI/Sign-Giorgio/src/mslm/models/imitator.py", line 92
        """
        
        def transformer_checkpoint(x):
        ~~~ <--- HERE
            return self.transformer(x, src_key_padding_mask=frames_padding_mask)
    
